In [0]:
df_bronze = spark.read.table("nyc_taxi_project.raw_data.bronze_taxi")
df_bronze.show(5)

In [0]:
df_bronze = spark.read.table("nyc_taxi_project.raw_data.bronze_taxi")
df_bronze.show(5)

In [0]:
df_bronze = spark.read.table("nyc_taxi_project.raw_data.bronze_taxi")
df_bronze.printSchema()
print("Bronze row count:", df_bronze.count())

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi_project.silver_data")

In [0]:
df_bronze = spark.read.table("nyc_taxi_project.raw_data.bronze_taxi")

In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, round as spark_round,
    hour, dayofweek, month, year, current_timestamp
)
from pyspark.sql.types import DoubleType, IntegerType

df_casted = df_bronze \
    .withColumn("VendorID",              col("VendorID").cast(IntegerType())) \
    .withColumn("passenger_count",       col("passenger_count").cast(DoubleType())) \
    .withColumn("trip_distance",         col("trip_distance").cast(DoubleType())) \
    .withColumn("pickup_longitude",      col("pickup_longitude").cast(DoubleType())) \
    .withColumn("pickup_latitude",       col("pickup_latitude").cast(DoubleType())) \
    .withColumn("dropoff_longitude",     col("dropoff_longitude").cast(DoubleType())) \
    .withColumn("dropoff_latitude",      col("dropoff_latitude").cast(DoubleType())) \
    .withColumn("RatecodeID",            col("RatecodeID").cast(IntegerType())) \
    .withColumn("payment_type",          col("payment_type").cast(IntegerType())) \
    .withColumn("fare_amount",           col("fare_amount").cast(DoubleType())) \
    .withColumn("extra",                 col("extra").cast(DoubleType())) \
    .withColumn("mta_tax",               col("mta_tax").cast(DoubleType())) \
    .withColumn("tip_amount",            col("tip_amount").cast(DoubleType())) \
    .withColumn("tolls_amount",          col("tolls_amount").cast(DoubleType())) \
    .withColumn("improvement_surcharge", col("improvement_surcharge").cast(DoubleType())) \
    .withColumn("total_amount",          col("total_amount").cast(DoubleType()))

In [0]:
df_casted = df_casted.dropna(subset=[
    "passenger_count", "trip_distance", "fare_amount",
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "total_amount"
])

In [0]:
df_silver = df_casted \
    .dropDuplicates() \
    .filter(col("passenger_count") > 0) \
    .filter(col("passenger_count") <= 6) \
    .filter(col("trip_distance") > 0) \
    .filter(col("trip_distance") < 100) \
    .filter(col("fare_amount") > 0) \
    .filter(col("fare_amount") < 500) \
    .filter(col("total_amount") > 0) \
    .filter(col("pickup_longitude").between(-74.25, -73.70)) \
    .filter(col("pickup_latitude").between(40.50, 40.92)) \
    .filter(col("dropoff_longitude").between(-74.25, -73.70)) \
    .filter(col("dropoff_latitude").between(40.50, 40.92)) \
    .withColumn("pickup_datetime",  to_timestamp(col("tpep_pickup_datetime"))) \
    .withColumn("dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime"))) \
    .withColumn(
        "trip_duration_minutes",
        spark_round(
            (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60,
            2
        )
    ) \
    .filter(col("trip_duration_minutes") > 0) \
    .filter(col("trip_duration_minutes") < 180) \
    .withColumn("pickup_hour",      hour("pickup_datetime")) \
    .withColumn("pickup_day",       dayofweek("pickup_datetime")) \
    .withColumn("pickup_month",     month("pickup_datetime")) \
    .withColumn("pickup_year",      year("pickup_datetime")) \
    .withColumn("silver_load_time", current_timestamp())

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df_silver.count())

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi_project.silver_data")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nyc_taxi_project.silver_data.silver_taxi")

print("Silver table saved!")

In [0]:
display(
    spark.sql("""
        SELECT pickup_year, pickup_month,
               COUNT(*)                    AS total_trips,
               ROUND(AVG(trip_distance),2) AS avg_distance_miles,
               ROUND(AVG(fare_amount),2)   AS avg_fare_usd,
               ROUND(AVG(tip_amount),2)    AS avg_tip_usd,
               ROUND(AVG(trip_duration_minutes),2) AS avg_duration_mins
        FROM nyc_taxi_project.silver_data.silver_taxi
        GROUP BY pickup_year, pickup_month
        ORDER BY pickup_year, pickup_month
    """)
)